# 

In [1]:
from pathlib import Path
from datetime import datetime as dt
import pandas as pd
import requests
from io import StringIO


In [2]:
now_ = dt.now().strftime('%Y%m%d')

In [3]:
DATA_DIR  = Path.cwd().parent / 'data'
assert DATA_DIR.exists()

In [5]:
# 1. Define the specific stations you want
STATIONS = [
    "HABs-BodegaMarineLab", "HABs-CalPolyPier",
    "HABs-Humboldt", "HABs-HumboldtSouthBay", "HABs-InnerTomalesBay",
    "HABs-MontereyWharf", "HABs-MorroBayBackBay", "HABs-MorroBayFrontBay",
    "HABs-NewportBeachPier", "HABs-SantaCruzWharf", "HABs-SantaMonicaPier",
    "HABs-ScrippsPier", "HABs-StearnsWharf", "HABs-TomalesBayMid-ChannelBuoy",
    "HABs-TomalesBayMouth", "HABs-TrinidadPier"
]

BASE_URL = "https://erddap.sccoos.org/erddap/tabledap"
START_TIME = "2024-03-06T00:00:00Z"

In [ ]:
def get_specific_hab_data():
    all_frames = []

    for dataset_id in STATIONS:
        # Construct the RESTful URL
        # We request all columns (?) and filter by time (>=START_TIME)
        # Using .csvp to avoid the units row that breaks pandas types
        url = f"{BASE_URL}/{dataset_id}.csvp?&time>={START_TIME}"
        
        print(f"Fetching: {dataset_id}...")
        try:
            response = requests.get(url)
            
            if response.status_code == 200:
                # Use StringIO to let pandas read the text response
                df = pd.read_csv(StringIO(response.text))
                if not df.empty:
                    # Optional: Add a column to keep track of the source dataset ID
                    df['source_dataset_id'] = dataset_id
                    all_frames.append(df)
            elif response.status_code == 404:
                print(f"  - No records found for {dataset_id} since March 6.")
            else:
                print(f"  - Error {response.status_code} for {dataset_id}")
                
        except Exception as e:
            print(f"  - Failed to download {dataset_id}: {e}")

    # Combine all results into one DataFrame
    if all_frames:
        # We use sort=False to maintain column order as much as possible
        combined_df = pd.concat(all_frames, ignore_index=True, sort=False)
        return combined_df
    else:
        print("No data found for the selected stations/timeframe.")
        return None

# Execute
df_final = get_specific_hab_data()

if df_final is not None:
    # Save the result
    output_filename = DATA_DIR / f"calhabmap_filtered_2024_{now_}.csv"
    df_final.to_csv(output_filename, index=False)
    print(f"\nSuccess! Compiled {len(df_final)} rows from {len(STATIONS)} stations.")
    print(f"Saved to: {output_filename}")

In [8]:
import pandas as pd
import requests
from io import StringIO
import re

# The specific stations you requested
STATIONS = [
    "HABs-BodegaMarineLabBuoy", "HABs-BodegaMarineLab", "HABs-CalPolyPier",
    "HABs-Humboldt", "HABs-HumboldtSouthBay", "HABs-InnerTomalesBay",
    "HABs-MontereyWharf", "HABs-MorroBayBackBay", "HABs-MorroBayFrontBay",
    "HABs-NewportBeachPier", "HABs-SantaCruzWharf", "HABs-SantaMonicaPier",
    "HABs-ScrippsPier", "HABs-StearnsWharf", "HABs-TomalesBayMid-ChannelBuoy",
    "HABs-TomalesBayMouth", "HABs-TrinidadPier"
]

BASE_URL = "https://erddap.sccoos.org/erddap"
START_TIME = "2024-03-06T00:00:00Z"

def get_station_names():
    """Fetches the Title for each dataset to use as the location_name."""
    print("Fetching station metadata...")
    # Get all dataset titles from the main index
    meta_url = f"{BASE_URL}/info/index.csv?datasetID,title"
    try:
        meta_df = pd.read_csv(meta_url, skiprows=[1])
        # Create a dictionary mapping Dataset ID -> Title
        # We clean the Title (e.g., "Harmful Algal Bloom Monitoring: Cal Poly Pier" -> "Cal Poly Pier")
        name_map = {}
        for _, row in meta_df.iterrows():
            if row['Dataset ID'] in STATIONS:
                # Use regex to strip the "Harmful Algal Bloom Monitoring: " prefix if present
                clean_name = re.sub(r'.*Monitoring:\s*', '', row['Title'])
                name_map[row['Dataset ID']] = clean_name
        return name_map
    except Exception as e:
        print(f"Warning: Could not fetch metadata index: {e}")
        return {}

def get_detailed_hab_data():
    station_names = get_station_names()
    all_frames = []

    for dataset_id in STATIONS:
        # Request .csvp (no units row)
        url = f"{BASE_URL}/tabledap/{dataset_id}.csvp?&time>={START_TIME}"
        
        print(f"Downloading: {dataset_id}...")
        try:
            response = requests.get(url)
            if response.status_code == 200:
                df = pd.read_csv(StringIO(response.text))
                if not df.empty:
                    # ADD THE MISSING LOCATION NAME
                    # We look up the name in our metadata map, defaulting to the ID if not found
                    df['location_name'] = station_names.get(dataset_id, dataset_id)
                    all_frames.append(df)
            else:
                print(f"  - No data found for {dataset_id}")
        except Exception as e:
            print(f"  - Error: {e}")

    if all_frames:
        combined_df = pd.concat(all_frames, ignore_index=True, sort=False)
        return combined_df
    return None

# Run and Save
df_final = get_detailed_hab_data()

if df_final is not None:
    print("\nFile saved: calhabmap_with_names.csv")
    print(df_final[['time (UTC)', 'location_name', 'Location_Code']].head())

Fetching station metadata...
Downloading: HABs-BodegaMarineLabBuoy...
Downloading: HABs-BodegaMarineLab...
Downloading: HABs-CalPolyPier...
Downloading: HABs-Humboldt...
Downloading: HABs-HumboldtSouthBay...
Downloading: HABs-InnerTomalesBay...
Downloading: HABs-MontereyWharf...
Downloading: HABs-MorroBayBackBay...
Downloading: HABs-MorroBayFrontBay...
Downloading: HABs-NewportBeachPier...
Downloading: HABs-SantaCruzWharf...
Downloading: HABs-SantaMonicaPier...
Downloading: HABs-ScrippsPier...
Downloading: HABs-StearnsWharf...
Downloading: HABs-TomalesBayMid-ChannelBuoy...
Downloading: HABs-TomalesBayMouth...
Downloading: HABs-TrinidadPier...

File saved: calhabmap_with_names.csv
             time (UTC)                                 location_name  \
0  2024-03-18T19:50:00Z  CalHABMAP - HABs Bodega Marine Lab Buoy data   
1  2024-05-16T18:30:00Z  CalHABMAP - HABs Bodega Marine Lab Buoy data   
2  2024-08-28T21:19:00Z  CalHABMAP - HABs Bodega Marine Lab Buoy data   
3  2024-09-30T16:11

In [7]:
df_final

,Location_Code,latitude (degrees_north),longitude (degrees_east),depth (m),SampleID,time (UTC),Temp (degree_C),Air_Temp (degree_C),Salinity,Chl_Volume_Filtered (mL),...,Prorocentrum_spp (cells/L),Pseudo_nitzschia_delicatissima_group (cells/L),Pseudo_nitzschia_seriata_group (cells/L),Ceratium_spp (cells/L),Cochlodinium_spp (cells/L),Gymnodinium_spp (cells/L),Other_Diatoms (cells/L),Other_Dinoflagellates (cells/L),Total_Phytoplankton (cells/L),location_name
0,BBB,38.31260,-123.08250,NaN,BML Buoy,2024-03-18T19:50:00Z,NaN,NaN,NaN,NaN,...,0.0,93.0,868.0,0.0,0.0,0.0,39133.0,155.0,40248.0,CalHABMAP - HABs Bodega Marine Lab Buoy data
1,BBB,38.31260,-123.08250,NaN,BML Buoy,2024-05-16T18:30:00Z,NaN,NaN,NaN,NaN,...,0.0,124.0,101.0,0.0,0.0,0.0,13966.0,77.0,14268.0,CalHABMAP - HABs Bodega Marine Lab Buoy data
2,BBB,38.31260,-123.08250,NaN,BML Buoy,2024-08-28T21:19:00Z,NaN,NaN,NaN,NaN,...,18.0,5200.0,0.0,36.0,0.0,0.0,1749.0,130.0,7183.0,CalHABMAP - HABs Bodega Marine Lab Buoy data
3,BBB,38.31260,-123.08250,NaN,BML Buoy,2024-09-30T16:11:00Z,NaN,NaN,NaN,NaN,...,81.0,0.0,0.0,11301.0,0.0,4.0,23.0,39.0,11518.0,CalHABMAP - HABs Bodega Marine Lab Buoy data
4,BBB,38.31260,-123.08250,NaN,BML Buoy,2025-01-17T18:37:00Z,NaN,NaN,NaN,NaN,...,1949.0,0.0,0.0,15167.0,0.0,0.0,2603.0,51.0,19783.0,CalHABMAP - HABs Bodega Marine Lab Buoy data
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
980,TP,41.05495,-124.14696,NaN,TRF_090,2024-06-20T23:45:00Z,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CalHABMAP - HABs Trinidad Pier data
981,TP,41.05495,-124.14696,NaN,TRF_091,2024-06-27T22:15:00Z,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CalHABMAP - HABs Trinidad Pier data
982,TP,41.05495,-124.14696,NaN,TRF_092,2024-07-10T23:50:00Z,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CalHABMAP - HABs Trinidad Pier data
983,TP,41.05495,-124.14696,NaN,TRF_093,2024-07-30T21:40:00Z,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CalHABMAP - HABs Trinidad Pier data


In [9]:
output_filename = DATA_DIR / f"calhabmap_filtered_2024_{now_}.csv"
df_final.to_csv(output_filename, index=False)